# x1_psoja-pmilho — Preços Mundiais de Soja e Milho (World Bank)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** preços de soja e milho, USD/t, mundial, mensal*  
**Conceito e Mecanismo Econômico:** Série temporal de preços internacionais de soja e milho (USD/tonelada métrica). Funciona como choque comum de demanda exógena aos municípios.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__prices__worldbank__mensal_robustez.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Selecionar price_soja e price_milho; explorar média anual e janela anterior à decisão de conversão.
> - Não somar preços mensais; definir calendário, transformação e eventual deflação/conversão monetária.

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`data`**: Data mensal da cotação internacional (YYYY-MM-DD).
- **`variavel`**: Commodity agrícola ('price_soja' ou 'price_milho').
- **`valor_original`**: Cotação bruta informada pelo Banco Mundial.
- **`unidade`**: Unidade física de medida ($/mt = USD por tonelada métrica).
- **`preco_nominal`**: Preço nominal em USD/t.
- **`ln_price`**: Logaritmo natural do preço (ln(P)).
- **`dln_price`**: Variação logarítmica mensal (retorno mensal).


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__prices__worldbank__mensal_robustez.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x1_psoja-pmilho`:
- **Atenção:** Selecionar price_soja e price_milho; explorar média anual e janela anterior à decisão de conversão.
- **Atenção:** Não somar preços mensais; definir calendário, transformação e eventual deflação/conversão monetária.


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month

# REGRA DO TXT:
# 'Selecionar price_soja e price_milho; explorar média anual e janela anterior à decisão de conversão.'
# 'Não somar preços mensais; definir calendário, transformação e eventual deflação/conversão monetária.'
df = df[df['variavel'].isin(['price_soja', 'price_milho'])].copy()

for col in ['preco_nominal', 'ln_price', 'dln_price']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.sort_values(['variavel', 'data'], inplace=True)
print(f"Total de registros mensais de soja e milho: {len(df)}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva dos Preços Mensais (USD/t):")
print(df.groupby('variavel')['preco_nominal'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

zeros = (df['preco_nominal'] == 0).sum()
nans = df['preco_nominal'].isna().sum()
print(f"Preços iguais a zero: {zeros} | Preços ausentes: {nans}")

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=df, x='preco_nominal', hue='variavel', kde=True, ax=axes[0], palette=['gold', 'darkorange'])
axes[0].set_title("Distribuição dos Preços Mundiais (USD/t)")

sns.boxplot(data=df, x='variavel', y='preco_nominal', ax=axes[1], palette=['gold', 'darkorange'])
axes[1].set_title("Boxplot dos Preços")

sns.lineplot(data=df, x='data', y='preco_nominal', hue='variavel', ax=axes[2], palette=['gold', 'darkorange'])
axes[2].set_title("Trajetória Histórica das Cotações")
plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
# Relação Soja x Milho no período 2001-2024
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
piv = df_painel.pivot(index='data', columns='variavel', values='preco_nominal').dropna()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=piv, x='price_soja', y='price_milho', color='darkgoldenrod')
plt.title(f"Correlação Soja vs Milho (2001-2024): r = {piv['price_soja'].corr(piv['price_milho']):.2f}")
plt.xlabel("Preço Soja (USD/t)")
plt.ylabel("Preço Milho (USD/t)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Comentário Econométrico:
# Preços mundiais são comuns a todos os municípios. Em painel com efeito fixo de ano (lambda_t),
# o efeito linear do preço é absorvido. É necessário interagir com exposição local (ex: Aptidão GAEZ).

## Fase 6: Feature Engineering — Harmonização e Interação Shift-Share de Preços

### 1. Step 1: Harmonização Temporal e Anualização com Janelas de Decisão
- **O problema dos dados mensais:** Preços do World Bank são mensais. A conversão de terras é decidida anualmente na safra.
- **Por que NUNCA somar preços mensais:** A soma de preços de janeiro a dezembro não tem significado econômico (um grão não vale a soma dos meses). A agregação correta é a **média**.
- **Duas métricas de anualização construídas:**
  1. **Média Anual do Ano Calendário ($t-1$):** A média dos 12 meses do ano anterior reflete o ambiente geral de preços em que o agricultor acumulou capital e planejou seus investimentos.
  2. **Janela Agronômica de Decisão Pré-Plantio (Março a Agosto $t-1$):** Em Mato Grosso, a semeadura da soja começa em setembro/outubro. A compra de fertilizantes, sementes, defensivos e os contratos a termo de venda futura são fechados entre março e agosto. Essa janela é o termômetro direto da decisão de investimento.

### 2. Step 2: Derivação de Variáveis e Interação Shift-Share (Efeito Fixo)
- **Razão de Preços Soja/Milho:** $RatioP_{t-1} = \frac{P_{soja, t-1}}{P_{milho, t-1}}$, mede o incentivo relativo de cultivo da soja versus milho safrinha.
- **O Desafio dos Efeitos Fixos de Ano ($\lambda_t$):** Em uma regressão em painel com efeito fixo temporal, o preço internacional linear é **completamente absorvido** porque varia apenas no tempo, sendo idêntico para todos os 141 municípios.
- **Solução Econométrica (Shift-Share / Bartik):** Interagir o choque macroeconômico exógeno de preço ($\ln P_{t-1}$) com uma medida local de exposição invariante no tempo (ex: Aptidão Biofísica GAEZ $i$):
  $$ChoqueLocal_{i, t} = \ln(P_{soja, t-1}) \times AptidaoGAEZ_i$$
  Isso gera variação espaço-temporal no painel ($i, t$), permitindo estimar a elasticidade local da conversão sem ser absorvido pelo efeito fixo de ano!


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Separar as duas commodities: soja e milho
piv_precos = df.pivot_table(index=['ano', 'mes'], columns='variavel', values='preco_nominal').reset_index()

# 2. MÉTRICA 1: Média Anual Calendário (12 meses)
p_anual = piv_precos.groupby('ano').agg(
    psoja_media_anual=('price_soja', 'mean'),
    pmilho_media_anual=('price_milho', 'mean')
).reset_index()

# 3. MÉTRICA 2: Janela Agronômica Pré-Plantio (Março a Agosto = meses 3 a 8)
# É nesta janela que o produtor compra sementes e decide se vai expandir lavoura para o próximo ano
janela_pre = piv_precos[piv_precos['mes'].isin([3, 4, 5, 6, 7, 8])].groupby('ano').agg(
    psoja_janela_preplantio=('price_soja', 'mean'),
    pmilho_janela_preplantio=('price_milho', 'mean')
).reset_index()

# Combinar as duas métricas em uma tabela única de preços anuais
precos_harmonizados = pd.merge(p_anual, janela_pre, on='ano', how='left')

# 4. Defasagem Temporal de 1 ano (t-1)
# O preço do ano passado é que explica a terra desmatada e preparada para o plantio deste ano
precos_harmonizados['psoja_lag1'] = precos_harmonizados['psoja_janela_preplantio'].shift(1)
precos_harmonizados['pmilho_lag1'] = precos_harmonizados['pmilho_janela_preplantio'].shift(1)

# Transformação Logarítmica dos preços defasados (elasticidade percentual)
precos_harmonizados['ln_psoja_lag1'] = np.log(precos_harmonizados['psoja_lag1'])
precos_harmonizados['ln_pmilho_lag1'] = np.log(precos_harmonizados['pmilho_lag1'])

# 5. Razão de Preços Soja / Milho (incentivo relativo entre as culturas)
precos_harmonizados['razao_psoja_pmilho_lag1'] = precos_harmonizados['psoja_lag1'] / precos_harmonizados['pmilho_lag1']

# 6. Código-esqueleto para interação Shift-Share no Painel Municipal:
# ChoquePreco_it = ln_psoja_lag1 * aptidao_gaez_i
precos_harmonizados['exemplo_choque_shift_share'] = precos_harmonizados['ln_psoja_lag1'] * 0.85

painel_precos = precos_harmonizados[(precos_harmonizados['ano'] >= 2001) & (precos_harmonizados['ano'] <= 2024)].copy()

print("Série de Preços Anualizados e Defasados (2001-2024):")
print(painel_precos[['ano', 'psoja_lag1', 'pmilho_lag1', 'ln_psoja_lag1', 'razao_psoja_pmilho_lag1']].tail(8))